In [3]:
import pickle
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

with open("portfolio_state.pkl", "rb") as f:
    state = pickle.load(f)

portfolio_value      = state["portfolio_value"]
portfolio_cost_basis = state["portfolio_cost_basis"]
invested_capital     = state["invested_capital"]
position_values      = state["position_values"]
ticker_asset_type    = state["ticker_asset_type"]
unrealised_pnl       = state["unrealised_pnl"]
realised_pnl         = state["realised_pnl"]
total_pnl            = state["total_pnl"]
summary_df           = state["summary_df"]

# DASHBOARD

In [4]:
# ---------------- 1. Portfolio value over time ----------------
fig_value = go.Figure()
fig_value.add_trace(go.Scatter(x=portfolio_value.index, y=portfolio_value.values,
                               name="Portfolio Value", line=dict(width=2)))
fig_value.add_trace(go.Scatter(x=portfolio_cost_basis.index, y=portfolio_cost_basis.values,
                               name="Cost Basis", line=dict(width=2)))
fig_value.add_trace(go.Scatter(x=invested_capital.index, y=invested_capital.values,
                               name="Invested Capital", line=dict(width=2)))
fig_value.update_layout(title="Portfolio Value Over Time",
                        xaxis_title="Date", yaxis_title="USD",
                        yaxis_tickprefix="$", yaxis_tickformat=",",
                        height=450, template="plotly_white")

# ---------------- 2. Current allocation ----------------
latest_positions = position_values.iloc[-1]
latest_positions = latest_positions[latest_positions > 0]
asset_type_values = latest_positions.groupby(ticker_asset_type.loc[latest_positions.index]).sum()
latest_pct = (latest_positions / latest_positions.sum()).sort_values()

fig_alloc = make_subplots(rows=1, cols=2,
                          specs=[[{"type": "pie"}, {"type": "bar"}]],
                          subplot_titles=("By Asset Type", "By Ticker (%)"))
fig_alloc.add_trace(go.Pie(labels=asset_type_values.index,
                           values=asset_type_values.values,
                           hole=0.3), row=1, col=1)
fig_alloc.add_trace(go.Bar(y=latest_pct.index, x=latest_pct.values,
                           orientation="h",
                           text=[f"{v:.1%}" for v in latest_pct.values],
                           textposition="outside",
                           marker_color=["#ee5253" if ticker_asset_type[t] == "Equity"
                                         else "#2e86de" for t in latest_pct.index]),
                   row=1, col=2)
fig_alloc.update_xaxes(tickformat=".0%", row=1, col=2)
fig_alloc.update_layout(title="Current Allocation", height=500,
                        showlegend=False, template="plotly_white")

# ---------------- 3. P&L breakdown chart ----------------
fig_pnl = go.Figure()
fig_pnl.add_trace(go.Scatter(x=unrealised_pnl.index, y=unrealised_pnl.values,
                             name="Unrealised P&L", line=dict(width=2)))
fig_pnl.add_trace(go.Scatter(x=realised_pnl.index, y=realised_pnl.values,
                             name="Realised P&L", line=dict(width=2)))
fig_pnl.add_trace(go.Scatter(x=total_pnl.index, y=total_pnl.values,
                             name="Total P&L", line=dict(width=3)))
fig_pnl.add_hline(y=0, line_color="black", line_width=1)
fig_pnl.update_layout(title="Portfolio P&L Breakdown",
                      xaxis_title="Date", yaxis_title="USD",
                      yaxis_tickprefix="$", yaxis_tickformat=",",
                      height=450, template="plotly_white")

# ---------------- 4. P&L summary cards ----------------
current_unrealised = unrealised_pnl.iloc[-1]
current_realised   = realised_pnl.iloc[-1]
current_total      = total_pnl.iloc[-1]
current_value      = portfolio_value.iloc[-1]

def card(label, value, is_pnl=True):
    color = "#27ae60" if (is_pnl and value >= 0) else ("#c0392b" if is_pnl else "#2c3e50")
    return f"""
    <div class="card">
        <div class="card-label">{label}</div>
        <div class="card-value" style="color: {color}">${value:,.2f}</div>
    </div>"""

summary_cards = (
    card("Portfolio Value", current_value, is_pnl=False) +
    card("Total P&L", current_total) +
    card("Unrealised P&L", current_unrealised) +
    card("Realised P&L", current_realised)
)

# ---------------- 5. Per-asset holdings table ----------------
display_cols = ["Ticker", "Shares", "Last price", "AC/share",
                "Market value (US$)", "Tot gain UNRL (US$)", "Tot gain UNRL (%)",
                "Realised gain (US$)"]
table_df = summary_df[display_cols].copy()
for col in ["Last price", "AC/share", "Market value (US$)",
            "Tot gain UNRL (US$)", "Realised gain (US$)"]:
    table_df[col] = table_df[col].map(lambda x: f"${x:,.2f}")
table_df["Tot gain UNRL (%)"] = table_df["Tot gain UNRL (%)"].map(lambda x: f"{x:.2%}")
table_df["Shares"] = table_df["Shares"].map(lambda x: f"{x:,.4f}")

holdings_table = table_df.to_html(index=False, classes="holdings", border=0)

# ---------------- Assemble HTML ----------------
html = f"""<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>Portfolio Dashboard</title>
<script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
<style>
body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
       background: #f5f6fa; margin: 0; padding: 20px; color: #2c3e50; }}
h1 {{ text-align: center; margin-bottom: 10px; }}
.timestamp {{ text-align: center; color: #7f8c8d; margin-bottom: 30px; }}
.cards {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
         gap: 15px; margin-bottom: 30px; }}
.card {{ background: white; padding: 20px; border-radius: 8px;
        box-shadow: 0 2px 4px rgba(0,0,0,0.05); text-align: center; }}
.card-label {{ font-size: 0.85em; color: #7f8c8d; text-transform: uppercase; }}
.card-value {{ font-size: 1.6em; font-weight: 600; margin-top: 8px; }}
.chart {{ background: white; padding: 15px; border-radius: 8px;
         box-shadow: 0 2px 4px rgba(0,0,0,0.05); margin-bottom: 20px; }}
.holdings {{ width: 100%; border-collapse: collapse; background: white; }}
.holdings th, .holdings td {{ padding: 10px; text-align: right; border-bottom: 1px solid #ecf0f1; }}
.holdings th {{ background: #34495e; color: white; text-align: center; }}
.holdings td:first-child, .holdings th:first-child {{ text-align: left; font-weight: 600; }}
.section-title {{ font-size: 1.2em; margin: 25px 0 10px; color: #34495e; }}
</style>
</head>
<body>
<h1>Portfolio Dashboard</h1>
<div class="timestamp">Generated {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")}</div>

<div class="cards">{summary_cards}</div>

<div class="chart" id="chart_value"></div>
<div class="chart" id="chart_alloc"></div>
<div class="chart" id="chart_pnl"></div>

<div class="section-title">Current Holdings</div>
<div class="chart">{holdings_table}</div>

<script>
Plotly.newPlot('chart_value', {fig_value.to_json()}.data, {fig_value.to_json()}.layout, {{responsive: true}});
Plotly.newPlot('chart_alloc', {fig_alloc.to_json()}.data, {fig_alloc.to_json()}.layout, {{responsive: true}});
Plotly.newPlot('chart_pnl', {fig_pnl.to_json()}.data, {fig_pnl.to_json()}.layout, {{responsive: true}});
</script>
</body>
</html>
"""

with open("index.html", "w", encoding="utf-8") as f:
    f.write(html)

print("Dashboard written to index.html")

Dashboard written to index.html
